# YOLO v8 → v11 benchmark: accuracy vs speed

Trains every size of YOLOv8, v9, v10 and YOLO11 on the same data with the same settings, then measures:

- **Accuracy** on the val split: mAP50-95, mAP50, mAP75, precision, recall, F1, and recall by object size (tiny persons are the hard part).
- **Speed** at batch 1: end-to-end latency (letterbox + network + NMS, how it runs on a drone) and network-only latency, mean and p95, plus FPS.
- **Cost**: parameters, GFLOPs at the training image size, weight file size, training time.

Input: `shard_*.npz` folders on Google Drive, `CapstoneFiles/data/rgb/<size>/{train,val}` (made by `src/detection/prepare_rgbtdroneperson.py`).
Weights and metrics go to Drive (`CapstoneFiles/yolo_bench/`), so a disconnected Colab session picks up where it stopped: reconnect and rerun all cells. Runtime → Change runtime type → GPU.

**Budget.** 21 models × 50 epochs is a lot. On a T4 at 256 px with ~5k images, the n/s models take minutes, the l/x models up to an hour each. Comment out models in `MODELS` to fit your GPU time.

In [ ]:
%pip install -q "ultralytics>=8.3.0" pandas matplotlib

from google.colab import drive
drive.mount("/content/drive")

## 1. Config

In [ ]:
import gc
import json
import shutil
import time
import traceback
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from scipy.optimize import linear_sum_assignment
from torchvision.ops import box_iou
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops

DRIVE = Path("/content/drive/MyDrive/CapstoneFiles")
# Sharded format of src/detection/prepare_*.py; {size} and {split} are filled in.
SHARD_DIR = str(DRIVE / "data/rgb/{size}/{split}")
OUT_DIR = DRIVE / "yolo_bench"                    # weights, metrics, plots: on Drive, survive disconnects
DATASET_DIR = Path("/content/yolo_datasets")      # PNG copies for ultralytics: local disk is much faster, rebuilt each session

IMG_SIZES = [128]                 # one entry per uploaded size folder, e.g. [128, 640] to also compare input resolution
SPLITS = ["train", "val"]         # "val" is used for evaluation

# Alternative to SHARD_DIR (set it to None): flat files in DATA_DIR, {size} and {split} filled in.
DATA_DIR = DRIVE / "data"
IMAGES_FILE = "{size}_{split}_images.npy"
LABELS_FILE = "{size}_{split}_labels.npz"

BOX_FORMAT = "cxcywh"             # how boxes are stored: "cxcywh" or "xyxy"; pixels or 0..1 is detected
CLASS_NAMES = ["person"]          # one name -> all source classes collapse into class 0

MODELS = {
    "YOLOv8":  ["yolov8n", "yolov8s", "yolov8m", "yolov8l", "yolov8x"],
    "YOLOv9":  ["yolov9t", "yolov9s", "yolov9m", "yolov9c", "yolov9e"],
    "YOLOv10": ["yolov10n", "yolov10s", "yolov10m", "yolov10b", "yolov10l", "yolov10x"],
    "YOLO11":  ["yolo11n", "yolo11s", "yolo11m", "yolo11l", "yolo11x"],
}

# Identical for every model, so differences come from the architecture only.
TRAIN = dict(epochs=50, patience=15, batch=64, workers=2, seed=0, deterministic=True)

SPEED_IMAGES = 50                 # val images cycled through during the latency test
SPEED_WARMUP = 30
SPEED_RUNS = 200
CONF = 0.25                       # confidence threshold for latency and size-bin recall (mAP uses the full PR curve)

DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
HALF = DEVICE == 0                # FP16 inference on CUDA, as you would deploy it
print("device:", torch.cuda.get_device_name(0) if DEVICE == 0 else DEVICE, "| fp16:", HALF)

METRICS_DIR = OUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Look at the data files
Check the key names and shapes below match what the loader expects.

In [ ]:
if SHARD_DIR:
    files = [p for size in IMG_SIZES for split in SPLITS
             for p in sorted(Path(SHARD_DIR.format(size=size, split=split)).glob("shard_*.npz"))[:1]]
else:
    files = sorted(DATA_DIR.glob("*.np[yz]"))
for p in files:
    if p.suffix == ".npy":
        a = np.load(p, mmap_mode="r")
        print(f"{p.name:40s} {a.dtype} {a.shape}")
    else:
        with np.load(p, allow_pickle=True) as z:
            print(f"{p.parent.name}/{p.name:20s} " + ", ".join(f"{k} {z[k].dtype}{z[k].shape}" for k in z.files))

## 3. Arrays → YOLO folder
ultralytics trains from image files, so each image size is written to `/content/yolo_datasets/<size>/` as PNG + txt labels (once per Colab session).

The label loader accepts:
- `boxes` + `box_counts` (+ optional `classes`): the format of `src/detection/prepare_*.py`;
- `labels`: an object array with one `(k, 4)` or `(k, 5)` array per image (`[cls,] box`);
- one key per image, in image order.

In [ ]:
def _rows(a):
    # one image's labels -> (k, 5) [cls, b0, b1, b2, b3]
    a = np.asarray(a, dtype=np.float32)
    if a.size == 0:
        return np.zeros((0, 5), np.float32)
    a = a.reshape(-1, a.shape[-1])
    return np.column_stack([np.zeros(len(a), np.float32), a]) if a.shape[1] == 4 else a[:, :5]


def read_labels(z, n_images):
    keys = set(z.files)
    if {"boxes", "box_counts"} <= keys:
        boxes, counts = z["boxes"], z["box_counts"]
        cls = z["classes"] if "classes" in keys else np.zeros(len(boxes))
        return np.split(np.column_stack([cls, boxes]).astype(np.float32), np.cumsum(counts)[:-1])
    if "labels" in keys:
        return [_rows(a) for a in z["labels"]]
    if len(keys) == n_images:
        return [_rows(z[k]) for k in z.files]
    raise ValueError(f"don't know this label layout: keys={sorted(keys)[:10]}, images={n_images}")


def load_split(size, split):
    # yields (images, labels) chunks: one chunk for flat files, one per shard otherwise
    if SHARD_DIR:
        shards = sorted(Path(SHARD_DIR.format(size=size, split=split)).glob("shard_*.npz"))
        if not shards:
            raise FileNotFoundError(SHARD_DIR.format(size=size, split=split))
        for s in shards:
            with np.load(s) as z:
                yield z["images"], read_labels(z, len(z["images"]))
    else:
        images = np.load(DATA_DIR / IMAGES_FILE.format(size=size, split=split), mmap_mode="r")
        with np.load(DATA_DIR / LABELS_FILE.format(size=size, split=split), allow_pickle=True) as z:
            labels = read_labels(z, len(images))
        assert len(labels) == len(images), f"{len(labels)} label sets for {len(images)} images"
        yield images, labels


def to_hwc_uint8(img):
    img = np.asarray(img)
    if img.ndim == 3 and img.shape[0] in (1, 3) and img.shape[-1] not in (1, 3):
        img = img.transpose(1, 2, 0)                  # CHW -> HWC
    if img.ndim == 3 and img.shape[2] == 1:
        img = img[:, :, 0]
    if img.dtype != np.uint8:
        img = (img * 255 if img.max() <= 1 else img).clip(0, 255).astype(np.uint8)
    return img


def to_yolo(rows, w, h, pixels):
    b = rows[:, 1:5].copy()
    if pixels:
        b /= np.array([w, h, w, h], np.float32)
    if BOX_FORMAT == "xyxy":
        b = np.column_stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2, b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]])
    b = b.clip(0, 1)
    cls = np.zeros(len(b)) if len(CLASS_NAMES) == 1 else rows[:, 0]
    keep = (b[:, 2] > 0) & (b[:, 3] > 0)
    return np.column_stack([cls[keep], b[keep]])


def build_yolo_dataset(size):
    root = DATASET_DIR / str(size)
    data_yaml = root / "data.yaml"
    if data_yaml.exists():                            # written last, so it marks a complete conversion
        return data_yaml
    shutil.rmtree(root, ignore_errors=True)
    for split in SPLITS:
        (root / "images" / split).mkdir(parents=True)
        (root / "labels" / split).mkdir(parents=True)
        i = 0
        for images, labels in load_split(size, split):
            pixels = max((r[:, 1:].max() for r in labels if len(r)), default=0) > 1.5
            for img, rows in zip(images, labels):
                img = to_hwc_uint8(img)
                Image.fromarray(img).save(root / "images" / split / f"{i:06d}.png")
                lines = [f"{int(c)} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in to_yolo(rows, img.shape[1], img.shape[0], pixels)]
                (root / "labels" / split / f"{i:06d}.txt").write_text("".join(lines))
                i += 1
        print(f"{size}px {split}: {i} images")
    data_yaml.write_text(yaml.safe_dump({
        "path": str(root),
        **{s: f"images/{s}" for s in SPLITS},
        "names": dict(enumerate(CLASS_NAMES)),
    }))
    return data_yaml


DATASETS = {size: build_yolo_dataset(size) for size in IMG_SIZES}

## 4. Sanity check
Object size in pixels at the model input decides what is detectable: below ~8 px a stride-8 head has about one grid cell per object.

In [ ]:
def read_yolo_labels(lbl_dir):
    out = {}
    for p in sorted(Path(lbl_dir).glob("*.txt")):
        out[p.stem] = np.array(p.read_text().split(), np.float32).reshape(-1, 5)
    return out


GT = {}  # size -> {stem: (k, 5) normalized val labels}, reused in section 8
for size, data_yaml in DATASETS.items():
    root = data_yaml.parent
    GT[size] = read_yolo_labels(root / "labels" / "val")
    for split in SPLITS:
        labels = read_yolo_labels(root / "labels" / split) if split != "val" else GT[size]
        all_b = np.concatenate(list(labels.values())) if labels else np.zeros((0, 5))
        side = np.sqrt(all_b[:, 3] * all_b[:, 4]) * size
        per_img = np.array([len(v) for v in labels.values()])
        print(f"{size}px {split:5s}: {len(labels)} images, {len(all_b)} boxes, "
              f"{per_img.mean():.1f}/image, {np.mean(per_img == 0):.1%} empty | "
              f"box side px p5/p50/p95 = {np.percentile(side, [5, 50, 95]).round(1)}, "
              f"under 8 px: {np.mean(side < 8):.0%}")

size = IMG_SIZES[0]
val_imgs = sorted((DATASETS[size].parent / "images" / "val").glob("*.png"))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, val_imgs[:: max(1, len(val_imgs) // 8)]):
    img = np.array(Image.open(p).convert("RGB"))
    ax.imshow(img)
    for _, cx, cy, w, h in GT[size][p.stem]:
        ax.add_patch(plt.Rectangle(((cx - w / 2) * size, (cy - h / 2) * size), w * size, h * size,
                                   fill=False, edgecolor="#eb6834", linewidth=1))
    ax.set_title(f"{p.stem}: {len(GT[size][p.stem])} boxes", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Train, evaluate, time one model
- **Training** resumes from `last.pt` if a previous session died mid-run.
- **mAP etc.** come from `model.val` on the val split with the best checkpoint.
- **Latency**: `e2e` is `model.predict` on one image (letterbox, network, NMS, result objects). YOLOv10 is NMS-free, so it gains here, not in `net`. `net` is the network forward pass alone on a fixed tensor. GPU work is synchronized around each call.

In [ ]:
def sync():
    if DEVICE == 0:
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


def timed(fn, n):
    t = []
    for _ in range(n):
        sync()
        t0 = time.perf_counter()
        fn()
        sync()
        t.append((time.perf_counter() - t0) * 1e3)
    return np.array(t)


@torch.no_grad()
def measure_speed(weights, imgsz, images):
    model = YOLO(weights)
    kw = dict(imgsz=imgsz, device=DEVICE, half=HALF, conf=CONF, verbose=False)
    k = iter(range(10**9))
    predict = lambda: model.predict(images[next(k) % len(images)], **kw)
    timed(predict, SPEED_WARMUP)
    e2e = timed(predict, SPEED_RUNS)

    net = model.predictor.model                       # fused, on device, fp16 if HALF
    p = next(net.parameters())
    x = torch.rand(1, 3, imgsz, imgsz, device=p.device, dtype=p.dtype)
    timed(lambda: net(x), SPEED_WARMUP)
    fwd = timed(lambda: net(x), SPEED_RUNS)
    return {
        "e2e_ms": e2e.mean(), "e2e_p95_ms": np.percentile(e2e, 95), "fps": 1e3 / e2e.mean(),
        "net_ms": fwd.mean(), "net_p95_ms": np.percentile(fwd, 95),
    }


def train(name, size, data_yaml):
    run_dir = OUT_DIR / "train" / f"{name}_{size}"
    info = run_dir / "train_info.json"
    if info.exists():
        return run_dir, json.loads(info.read_text())
    last = run_dir / "weights" / "last.pt"
    used_amp = lambda: yaml.safe_load((run_dir / "args.yaml").read_text())["amp"]
    fresh = lambda amp: YOLO(f"{name}.pt").train(data=str(data_yaml), imgsz=size, device=DEVICE, amp=amp,
                                                 project=str(run_dir.parent), name=run_dir.name, exist_ok=True, **TRAIN)
    t0 = time.time()
    try:
        if last.exists():
            print(f"resuming {run_dir.name} (train time below covers this session only)")
            YOLO(last).train(resume=True)
        else:
            fresh(True)
    except RuntimeError as e:
        # fp16 (AMP) can overflow on these tiny boxes; fp32 is slower but stable
        if "NaN" not in str(e) or not used_amp():
            raise
        print(f"{run_dir.name}: loss went NaN under AMP, retraining from scratch in fp32")
        shutil.rmtree(run_dir)
        t0 = time.time()
        fresh(False)
    d = {"train_hours": (time.time() - t0) / 3600,
         "epochs_run": len(pd.read_csv(run_dir / "results.csv")),
         "amp": used_amp()}
    info.write_text(json.dumps(d))
    return run_dir, d


def benchmark_one(family, name, size, data_yaml, speed_images):
    run_dir, train_info = train(name, size, data_yaml)
    best = run_dir / "weights" / "best.pt"

    model = YOLO(best)
    params = sum(p.numel() for p in model.model.parameters())
    gflops = get_flops(model.model, size)
    m = model.val(data=str(data_yaml), imgsz=size, batch=TRAIN["batch"], device=DEVICE, half=HALF,
                  split="val", plots=False, verbose=False, project=str(OUT_DIR / "val"), name=run_dir.name, exist_ok=True)
    p, r = float(m.box.mp), float(m.box.mr)
    del model

    return {
        "run": run_dir.name, "family": family, "model": name, "imgsz": size,
        "mAP50_95": float(m.box.map), "mAP50": float(m.box.map50), "mAP75": float(m.box.map75),
        "precision": p, "recall": r, "f1": 2 * p * r / (p + r + 1e-9),
        **measure_speed(best, size, speed_images),
        "val_ms_per_img": sum(m.speed.values()),      # at val batch size, for reference
        "params_M": params / 1e6, "gflops": gflops, "weights_MB": best.stat().st_size / 1e6,
        **train_info, "weights": str(best),
    }

## 6. Run everything
Safe to rerun: finished runs are skipped, failures (e.g. out of memory on the x models) are logged and the loop moves on.

In [ ]:
failures = {}
for size, data_yaml in DATASETS.items():
    val_paths = sorted((data_yaml.parent / "images" / "val").glob("*.png"))
    speed_images = [cv2.imread(str(p)) for p in val_paths[:SPEED_IMAGES]]
    for family, names in MODELS.items():
        for name in names:
            out = METRICS_DIR / f"{name}_{size}.json"
            if out.exists():
                print(f"skip {out.stem} (done)")
                continue
            print(f"\n===== {family} {name} @ {size}px =====")
            try:
                row = benchmark_one(family, name, size, data_yaml, speed_images)
                out.write_text(json.dumps(row, indent=2, default=float))
                print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})
            except Exception as e:
                failures[f"{name}_{size}"] = repr(e)
                traceback.print_exc()
            finally:
                gc.collect()
                if DEVICE == 0:
                    torch.cuda.empty_cache()
print("failures:", failures or "none")

## 7. Results
A model is on the **Pareto front** if no other model at the same image size is both faster (e2e) and more accurate (mAP50-95). Those are the only ones worth picking; the rest are dominated.

In [ ]:
df = pd.DataFrame([json.loads(p.read_text()) for p in sorted(METRICS_DIR.glob("*.json"))])
order = {m: i for i, m in enumerate(n for ns in MODELS.values() for n in ns)}
df = df.sort_values(["imgsz", "model"], key=lambda c: c.map(order) if c.name == "model" else c).reset_index(drop=True)


df["pareto"] = False
for _, g in df.groupby("imgsz"):
    g = g.sort_values("e2e_ms")
    df.loc[g.index, "pareto"] = g["mAP50_95"] > g["mAP50_95"].cummax().shift(fill_value=-1)
df.to_csv(OUT_DIR / "results.csv", index=False)

cols = ["family", "model", "imgsz", "mAP50_95", "mAP50", "recall", "precision", "f1",
        "e2e_ms", "e2e_p95_ms", "fps", "net_ms", "params_M", "gflops", "weights_MB", "train_hours", "epochs_run", "pareto"]
(df[cols].style
    .format({c: "{:.3f}" for c in ["mAP50_95", "mAP50", "recall", "precision", "f1"]}
            | {c: "{:.1f}" for c in ["e2e_ms", "e2e_p95_ms", "fps", "net_ms", "params_M", "gflops", "weights_MB"]}
            | {"train_hours": "{:.2f}"})
    .background_gradient(subset=["mAP50_95", "mAP50", "recall"], cmap="Blues")
    .background_gradient(subset=["e2e_ms", "net_ms"], cmap="Greys")
    .apply(lambda r: ["font-weight: bold" if r["pareto"] else ""] * len(r), axis=1))

In [ ]:
# Best model per family at each size, and the fastest one within 1 mAP point of the overall best.
for size, g in df.groupby("imgsz"):
    print(f"--- {size}px ---")
    print(g.loc[g.groupby("family")["mAP50_95"].idxmax(), ["family", "model", "mAP50_95", "e2e_ms", "params_M"]].to_string(index=False))
    near = g[g["mAP50_95"] >= g["mAP50_95"].max() - 0.01].nsmallest(1, "e2e_ms").iloc[0]
    print(f"fastest within 0.01 mAP50-95 of best: {near['model']} ({near['mAP50_95']:.3f}, {near['e2e_ms']:.1f} ms)\n")

### Accuracy vs cost
One panel per family: that family in color with its sizes connected, the other families in grey for context. The dashed line is the Pareto front across all families.

In [ ]:
HILITE, CONTEXT, FRONT = "#2a78d6", "#c9c8c2", "#52514e"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True})


def tradeoff_grid(g, x, xlabel, logx=True):
    fams = list(MODELS)
    fig, axes = plt.subplots(1, len(fams), figsize=(4.2 * len(fams), 4), sharex=True, sharey=True)
    front = g[g["pareto"]].sort_values(x)
    for ax, fam in zip(np.atleast_1d(axes), fams):
        for other, o in g[g["family"] != fam].groupby("family"):
            ax.plot(o[x], o["mAP50_95"], "-o", color=CONTEXT, lw=1, ms=4, zorder=1)
        ax.plot(front[x], front["mAP50_95"], "--", color=FRONT, lw=1, zorder=2)
        f = g[g["family"] == fam]
        ax.plot(f[x], f["mAP50_95"], "-o", color=HILITE, lw=2, ms=8, mec="white", mew=2, zorder=3)
        for _, r in f.iterrows():
            ax.annotate(r["model"].replace("yolo", "").replace("v", ""), (r[x], r["mAP50_95"]),
                        textcoords="offset points", xytext=(5, -10), fontsize=8, color="#52514e")
        ax.set_title(fam, loc="left", fontsize=11)
        ax.set_xlabel(xlabel)
        if logx:
            ax.set_xscale("log")
    np.atleast_1d(axes)[0].set_ylabel("mAP50-95 (val)")
    fig.suptitle(f"{g['imgsz'].iloc[0]} px input", x=0.01, ha="left", fontsize=12)
    plt.tight_layout()
    return fig


for size, g in df.groupby("imgsz"):
    for x, label in [("e2e_ms", f"end-to-end latency, ms (batch 1, {'fp16' if HALF else 'fp32'})"),
                     ("gflops", "GFLOPs"), ("params_M", "parameters, M")]:
        tradeoff_grid(g, x, label).savefig(OUT_DIR / f"tradeoff_{x}_{size}.png", dpi=150, bbox_inches="tight")
        plt.show()

### Training curves
Val mAP50-95 per epoch. Shade goes light → dark from the smallest to the largest model in each family.

In [ ]:
for size in IMG_SIZES:
    fams = list(MODELS)
    fig, axes = plt.subplots(1, len(fams), figsize=(4.2 * len(fams), 3.6), sharey=True)
    for ax, fam in zip(np.atleast_1d(axes), fams):
        names = MODELS[fam]
        shades = plt.cm.Blues(np.linspace(0.35, 0.95, len(names)))
        for name, c in zip(names, shades):
            csv = OUT_DIR / "train" / f"{name}_{size}" / "results.csv"
            if csv.exists():
                r = pd.read_csv(csv)
                r.columns = r.columns.str.strip()
                ax.plot(r["epoch"], r["metrics/mAP50-95(B)"], color=c, lw=2, label=name)
        ax.set_title(fam, loc="left", fontsize=11)
        ax.set_xlabel("epoch")
        ax.legend(fontsize=8, frameon=False)
    np.atleast_1d(axes)[0].set_ylabel("val mAP50-95")
    plt.tight_layout()
    fig.savefig(OUT_DIR / f"curves_{size}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Recall by object size
mAP is dominated by whatever object size is most common. This splits recall by box side in pixels at the model input, at `CONF` and IoU ≥ 0.5 with one-to-one matching. If recall collapses in the smallest bins for every model, the fix is input resolution or tiling, not a bigger model.

In [ ]:
SIZE_BINS = [0, 2, 4, 8, 16, 32, np.inf]
BIN_NAMES = ["<2", "2-4", "4-8", "8-16", "16-32", "32+"]


def recall_by_size(weights, size, iou_thr=0.5):
    model = YOLO(weights)
    root = DATASETS[size].parent
    hit = np.zeros(len(BIN_NAMES))
    tot = np.zeros(len(BIN_NAMES))
    paths = sorted((root / "images" / "val").glob("*.png"))
    for i in range(0, len(paths), 64):
        for res in model.predict([str(p) for p in paths[i:i + 64]], imgsz=size, device=DEVICE, half=HALF,
                                 conf=CONF, verbose=False, stream=True):
            gt = GT[size][Path(res.path).stem]
            if not len(gt):
                continue
            cx, cy, w, h = gt[:, 1:].T
            gt_xyxy = torch.tensor(np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1))
            b = np.digitize(np.sqrt(w * h) * size, SIZE_BINS) - 1
            np.add.at(tot, b, 1)
            pred = res.boxes.xyxyn.float().cpu()
            if len(pred):
                iou = box_iou(gt_xyxy, pred).numpy()
                gi, pi = linear_sum_assignment(-iou)
                matched = gi[iou[gi, pi] >= iou_thr]
                np.add.at(hit, b[matched], 1)
    del model
    return dict(zip(BIN_NAMES, hit / np.maximum(tot, 1))), dict(zip(BIN_NAMES, tot.astype(int)))


rows = []
for _, r in df.iterrows():
    rec, counts = recall_by_size(r["weights"], r["imgsz"])
    rows.append({"model": r["model"], "imgsz": r["imgsz"], **rec})
size_df = pd.DataFrame(rows)
size_df.to_csv(OUT_DIR / "recall_by_size.csv", index=False)
print("GT boxes per bin (last size):", counts)
size_df.style.format({b: "{:.2f}" for b in BIN_NAMES}).background_gradient(subset=BIN_NAMES, cmap="Blues", vmin=0, vmax=1)

## 9. Results on Drive
`CapstoneFiles/yolo_bench/`: `results.csv`, `recall_by_size.csv`, the plots, `train/<run>/weights/best.pt` for every model.